In [12]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import os
os.chdir(r'C:\Users\benjo\Documents\Projects\ecg-risk-stratification')
print(os.getcwd())

C:\Users\benjo\Documents\Projects\ecg-risk-stratification


Imports and Data Extraction

In [14]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/ptbxl_database.csv', index_col= 'ecg_id')
df.head()
df.info()
df.columns

<class 'pandas.DataFrame'>
Index: 21799 entries, 1 to 21837
Data columns (total 27 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   patient_id                    21799 non-null  float64
 1   age                           21799 non-null  float64
 2   sex                           21799 non-null  int64  
 3   height                        6974 non-null   float64
 4   weight                        9421 non-null   float64
 5   nurse                         20326 non-null  float64
 6   site                          21782 non-null  float64
 7   device                        21799 non-null  str    
 8   recording_date                21799 non-null  str    
 9   report                        21799 non-null  str    
 10  scp_codes                     21799 non-null  str    
 11  heart_axis                    13331 non-null  str    
 12  infarction_stadium1           5612 non-null   str    
 13  infarction_stadiu

Index(['patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site',
       'device', 'recording_date', 'report', 'scp_codes', 'heart_axis',
       'infarction_stadium1', 'infarction_stadium2', 'validated_by',
       'second_opinion', 'initial_autogenerated_report', 'validated_by_human',
       'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems',
       'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr'],
      dtype='str')

In [15]:
df[
    [
        "patient_id",
        "age",
        "sex",
        "recording_date",
        "report",
        "scp_codes",
        "strat_fold",
        "filename_lr",
        "filename_hr",
    ]
].head(10)

,patient_id,age,sex,recording_date,report,scp_codes,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,
1,15709.0,56.0,1,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,"{'NORM': 80.0, 'SBRAD': 0.0}",2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,1984-11-15 12:49:10,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,1984-11-15 13:44:57,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,1984-11-17 10:43:15,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",4,records100/00000/00005_lr,records500/00000/00005_hr
6,19005.0,18.0,1,1984-11-28 13:32:13,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",4,records100/00000/00006_lr,records500/00000/00006_hr
7,16193.0,54.0,0,1984-11-28 13:32:22,"sinusrhythmus linkstyp t abnormal, wahrscheinl...","{'NORM': 100.0, 'SR': 0.0}",7,records100/00000/00007_lr,records500/00000/00007_hr
8,11275.0,48.0,0,1984-12-01 14:49:52,sinusrhythmus linkstyp qrs(t) abnormal infe...,"{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}",9,records100/00000/00008_lr,records500/00000/00008_hr
9,18792.0,55.0,0,1984-12-08 09:44:43,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",10,records100/00000/00009_lr,records500/00000/00009_hr


In [16]:
df[
    ['scp_codes']
].head(10)


,scp_codes
ecg_id,
1,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}"
2,"{'NORM': 80.0, 'SBRAD': 0.0}"
3,"{'NORM': 100.0, 'SR': 0.0}"
4,"{'NORM': 100.0, 'SR': 0.0}"
5,"{'NORM': 100.0, 'SR': 0.0}"
6,"{'NORM': 100.0, 'SR': 0.0}"
7,"{'NORM': 100.0, 'SR': 0.0}"
8,"{'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}"
9,"{'NORM': 100.0, 'SR': 0.0}"


Converrting the SCP code table into a dict from the entries of strings as seen above

In [17]:
import ast 

df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

Extraction of ECG Data// File Names

In [18]:
from src.data_extraction import load_ecg

ecg_path = 'data/'
signal,meta,row = load_ecg(1,df,ecg_path)

'''print(signal.shape)
print(meta["fs"])
print(meta["sig_name"])
print(row["scp_codes"])
print(row["report"]) TESTING FUNCTION''' 

'print(signal.shape)\nprint(meta["fs"])\nprint(meta["sig_name"])\nprint(row["scp_codes"])\nprint(row["report"]) TESTING FUNCTION'

Information for the scp data

In [19]:
scp = pd.read_csv('data/scp_statements.csv',index_col=0)
scp[["description", "diagnostic", "diagnostic_class", "diagnostic_subclass"]].head(20)

,description,diagnostic,diagnostic_class,diagnostic_subclass
NDT,non-diagnostic T abnormalities,1.0,STTC,STTC
NST_,non-specific ST changes,1.0,STTC,NST_
DIG,digitalis-effect,1.0,STTC,STTC
LNGQT,long QT-interval,1.0,STTC,STTC
NORM,normal ECG,1.0,NORM,NORM
IMI,inferior myocardial infarction,1.0,MI,IMI
ASMI,anteroseptal myocardial infarction,1.0,MI,AMI
LVH,left ventricular hypertrophy,1.0,HYP,LVH
LAFB,left anterior fascicular block,1.0,CD,LAFB/LPFB
ISC_,non-specific ischemic,1.0,STTC,ISC_


Creating the Superclass label for classification of major diagnosis

In [20]:
from src.data_extraction import add_superclass_col

df = add_superclass_col(df, scp)


In [23]:
from src.data_extraction import add_binary_col

df = add_binary_col(df)
binary_df = df.dropna(subset=["label"])
binary_df["label"] = binary_df["label"].astype(int)

binary_df["label"].value_counts()

label
0    9069
1    5235
Name: count, dtype: int64

Save the processed metadata df

In [25]:
df.to_pickle("data/ptbxl_metadata_processed.pkl")
binary_df.to_pickle("data/ptbxl_binary_metadata.pkl")

Plotting and inspecting 2 ecg signals (norm and sttc)

In [ ]:
normal_id = binary_df[binary_df["label"] == 0].index[0]
sttc_id = binary_df[binary_df["label"] == 1].index[0]
print("Normal ECG ID:", normal_id)
print("STTC ECG ID:", sttc_id)

normal_signal, normal_meta, normal_row = load_ecg(normal_id, df, "data")
sttc_signal, sttc_meta, sttc_row = load_ecg(sttc_id, df, "data")

